In [0]:
fato_acidente = spark.table("prf_acidentes1.gold.fato_acidente")
dim_tempo = spark.table("prf_acidentes1.gold.dim_tempo")
dim_local = spark.table("prf_acidentes1.gold.dim_local")
dim_causa = spark.table("prf_acidentes1.gold.dim_causa")
dim_condicao = spark.table("prf_acidentes1.gold.dim_condicao")
dim_classificacao = spark.table("prf_acidentes1.gold.dim_classificacao")

print("Tabelas carregadas com sucesso.")

Tabelas carregadas com sucesso.


Pergunta 1: Ranking de BRs por acidentes e gravidade

In [0]:
from pyspark.sql import functions as F

df_p1 = (
    fato_acidente
    .join(dim_local, on="id_local", how="left")
    .filter(F.col("br") != 0)
    .groupBy("br")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves"),
        F.sum("feridos_leves").alias("total_feridos_leves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("qtd_acidentes"))
)

display(df_p1.limit(15))

br,qtd_acidentes,total_mortos,total_feridos_graves,total_feridos_leves,indice_gravidade
101,37426,2155,9624,33175,0.0576
116,33303,2270,7423,30031,0.0682
381,10252,606,2351,10002,0.0591
40,10124,616,2415,9657,0.0608
153,8077,793,2216,6998,0.0982
163,7339,698,1953,5771,0.0951
364,6579,544,1867,5381,0.0827
277,6194,495,1579,5467,0.0799
376,5326,372,1371,4455,0.0698
262,5197,463,1850,4473,0.0891


**Interpretação — Resposta à Pergunta 1**

BR-101 e BR-116 dominam disparado em volume de acidentes (37.426 e 33.303 respectivamente), muito à frente da 3ª colocada (BR-381, 10.252). Isso é coerente com a realidade: são as duas rodovias federais mais extensas e movimentadas do país, cortando praticamente todo o litoral (101) e o eixo Sul-Sudeste (116).

Porém, volume alto não significa maior risco por acidente. Olhando o índice de gravidade (mortos por acidente):

BR-316 tem o maior índice (0,1685) — quase 3x maior que a BR-101 (0,0576), mesmo tendo 10x menos acidentes

BR-230 (0,1082) e BR-153 (0,0982) também se destacam como proporcionalmente mais letais

Isso sugere uma conclusão importante: BR-101 e BR-116 precisam de atenção por volume, mas BR-316, BR-230 e BR-153 merecem atenção por letalidade por ocorrência — possivelmente por características da via (menos duplicada, mais trechos rurais/sinuosos), algo que podemos cruzar depois com dim_condicao.

Célula 3 — Pergunta 2: Gravidade por fase do dia

In [0]:
df_p2 = (
    fato_acidente
    .join(dim_tempo, on="id_tempo", how="left")
    .groupBy("fase_dia")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("indice_gravidade"))
)

display(df_p2)

fase_dia,qtd_acidentes,total_mortos,total_feridos_graves,indice_gravidade
Amanhecer,10250,1397,2965,0.1363
Plena Noite,73595,8534,22494,0.116
Anoitecer,11624,894,3273,0.0769
Pleno dia,117982,7005,30588,0.0594


**Interpretação — Resposta à Pergunta 2**

Há uma relação clara: existe sim relação entre horário/fase do dia e gravidade.

"Pleno dia" concentra o maior volume de acidentes (117.982, ~55% do total) — esperado, já que é quando há mais tráfego — mas é a fase com menor gravidade proporcional (0,0594 mortos/acidente).

"Amanhecer" tem o maior índice de gravidade (0,1363) — mais que o dobro do "Pleno dia" — apesar de ter bem menos acidentes em volume absoluto (10.250). Isso é um padrão bem documentado na literatura de segurança viária: acidentes de madrugada/amanhecer tendem a ser mais graves por fatores como sonolência, velocidade mais alta (menos tráfego) e menor visibilidade.

"Plena Noite" também tem gravidade alta (0,116), reforçando a hipótese de que o período noturno/madrugada é desproporcionalmente mais perigoso, mesmo com menos acidentes.

Conclusão: volume de acidentes e gravidade seguem padrões opostos ao longo do dia — políticas de segurança poderiam priorizar fiscalização/iluminação em horários de baixo movimento mas alto risco (madrugada/amanhecer), não só nos horários de pico.

Pergunta 3: "Quais são as principais causas de acidentes e como se relacionam com o número de vítimas?"

In [0]:
df_p3 = (
    fato_acidente
    .join(dim_causa, on="id_causa", how="left")
    .groupBy("causa_acidente")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves"),
        F.sum("feridos_leves").alias("total_feridos_leves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .orderBy(F.desc("qtd_acidentes"))
)

display(df_p3.limit(15))

causa_acidente,qtd_acidentes,total_mortos,total_feridos_graves,total_feridos_leves,indice_gravidade
Reação tardia ou ineficiente do condutor,31574,1838,7489,29086,0.0582
Ausência de reação do condutor,31459,2288,8084,26300,0.0727
Acessar a via sem observar a presença dos outros veículos,20380,1205,6850,18569,0.0591
Condutor deixou de manter distância do veículo da frente,13145,284,2536,12302,0.0216
Velocidade Incompatível,12675,1382,4036,11763,0.109
Manobra de mudança de faixa,12142,688,2866,11774,0.0567
Ingestão de álcool pelo condutor,11138,593,2493,6880,0.0532
Demais falhas mecânicas ou elétricas,9715,198,1195,5860,0.0204
Transitar na contramão,7300,2743,4183,6255,0.3758
Condutor Dormindo,6346,590,1862,6207,0.093


**Interpretação — Resposta à Pergunta 3**

As causas mais frequentes ("Reação tardia", "Ausência de reação", "Acessar via sem observar") têm gravidade relativamente moderada (0,058-0,073), mas volume altíssimo — juntas somam quase 40% dos acidentes e são as maiores responsáveis pelo total absoluto de vítimas.

Já as causas mais raras em volume, mas extremamente mais letais:

-"Transitar na contramão" — índice de gravidade de 0,3758, mais de 6x maior que a média das causas mais comuns. Só 7.300 acidentes, mas 2.743 mortos — quase tantos mortos quanto a BR-101 inteira (que tem 5x mais acidentes)

-"Ultrapassagem Indevida" — 0,2259, quase 4x mais letal que a média

Conclusão de negócio: fiscalização de volume (reação do condutor, distração) previne o maior número absoluto de acidentes, mas ações contra direção na contramão e ultrapassagens indevidas têm potencial de salvar desproporcionalmente mais vidas por intervenção, mesmo afetando menos ocorrências.

Pergunta 4: "Há diferença nos padrões de acidentes entre dias de semana e finais de semana?"

In [0]:
df_p4 = (
    fato_acidente
    .join(dim_tempo, on="id_tempo", how="left")
    .groupBy("flag_fim_de_semana")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
)

# Normalizando por número de dias (2 dias de FDS vs 5 dias de semana, por semana)
display(df_p4)

flag_fim_de_semana,qtd_acidentes,total_mortos,total_feridos_graves,indice_gravidade
true,68774,6838,20764,0.0994
false,144677,10992,38556,0.076


In [0]:
df_dias_distintos = (
    dim_tempo
    .select("data_inversa", "flag_fim_de_semana")
    .distinct()
    .groupBy("flag_fim_de_semana")
    .agg(F.count("data_inversa").alias("qtd_dias"))
)

df_p4_normalizado = df_p4.join(df_dias_distintos, on="flag_fim_de_semana", how="left") \
    .withColumn("media_acidentes_por_dia", F.round(F.col("qtd_acidentes") / F.col("qtd_dias"), 2))

display(df_p4_normalizado)

flag_fim_de_semana,qtd_acidentes,total_mortos,total_feridos_graves,indice_gravidade,qtd_dias,media_acidentes_por_dia
true,68774,6838,20764,0.0994,313,219.73
false,144677,10992,38556,0.076,783,184.77


**Interpretação — Resposta à Pergunta 4**

Sim, existe diferença clara nos dois indicadores:

Volume normalizado por dia: finais de semana têm ~19% mais acidentes por dia (219,73 vs 184,77) do que dias úteis — mesmo com menos tráfego de trabalho, provavelmente por viagens de lazer/turismo em rodovias federais.

Gravidade: finais de semana também são ~31% mais letais por acidente (índice 0,0994 vs 0,076) — coerente com fatores já conhecidos como maior consumo de álcool, velocidades mais altas em viagens de lazer, e maior cansaço em deslocamentos longos.

Conclusão: finais de semana são desproporcionalmente mais perigosos tanto em frequência quanto em gravidade — um padrão relevante para políticas de fiscalização reforçada às sextas/sábados/domingos.

Pergunta 5 : "Existe relação entre a quantidade de veículos envolvidos e a gravidade do acidente?"

In [0]:
df_p5 = (
    fato_acidente
    .groupBy("veiculos")
    .agg(
        F.count("id_acidente").alias("qtd_acidentes"),
        F.sum("mortos").alias("total_mortos"),
        F.sum("feridos_graves").alias("total_feridos_graves")
    )
    .withColumn("indice_gravidade", F.round(F.col("total_mortos") / F.col("qtd_acidentes"), 4))
    .filter(F.col("qtd_acidentes") >= 30)  # remove categorias muito raras (ruído estatístico)
    .orderBy("veiculos")
)

display(df_p5)

veiculos,qtd_acidentes,total_mortos,total_feridos_graves,indice_gravidade
1,70529,2755,14588,0.0391
2,102190,8478,30795,0.083
3,25266,3482,8268,0.1378
4,9016,1563,2990,0.1734
5,3724,841,1440,0.2258
6,1298,260,548,0.2003
7,639,193,300,0.302
8,327,90,125,0.2752
9,161,58,89,0.3602
10,95,24,63,0.2526


**Interpretação — Resposta à Pergunta 5**

Sim, existe uma relação clara e quase monotônica: quanto mais veículos envolvidos, maior a gravidade do acidente.

Acidentes com 1 veículo (saída de pista, capotamento sozinho) têm o menor índice (0,0391)

Acidentes com 2 veículos já dobram a gravidade (0,083)

A partir de 5+ veículos, o índice ultrapassa 0,20 — mais de 5x a gravidade de um acidente com 1 veículo só

O pico em 11 veículos (0,4211) é estatisticamente mais instável (apenas 57 casos), mas a tendência geral é muito consistente até ali

Conclusão: colisões múltiplas (engavetamentos) são desproporcionalmente mais letais que acidentes isolados, reforçando a importância de distância segura entre veículos (aliás, uma das causas mais frequentes que já vimos na Pergunta 3).

In [0]:
display(spark.table("prf_acidentes1.gold.dim_tempo").limit(10))

id_tempo,data_inversa,ano,mes,dia,dia_semana,horario,hora,fase_dia,flag_fim_de_semana
0,2023-01-01,2023,1,1,domingo,20:00:00,20,Plena Noite,true
1,2023-01-01,2023,1,1,domingo,21:54:00,21,Plena Noite,true
2,2023-01-02,2023,1,2,segunda-feira,09:00:00,9,Pleno dia,false
3,2023-01-02,2023,1,2,segunda-feira,14:00:00,14,Pleno dia,false
4,2023-01-02,2023,1,2,segunda-feira,15:16:00,15,Pleno dia,false
5,2023-01-02,2023,1,2,segunda-feira,17:00:00,17,Pleno dia,false
6,2023-01-03,2023,1,3,terça-feira,06:55:00,6,Amanhecer,false
7,2023-01-03,2023,1,3,terça-feira,08:15:00,8,Pleno dia,false
8,2023-01-03,2023,1,3,terça-feira,09:20:00,9,Pleno dia,false
9,2023-01-03,2023,1,3,terça-feira,23:40:00,23,Plena Noite,false


In [0]:
display(spark.table("prf_acidentes1.gold.dim_local").limit(10))

id_local,uf,br,km,municipio,regional,delegacia,uop,latitude,longitude
0,SC,101,134.0,BALNEARIO CAMBORIU,SPRF-SC,DEL04-SC,UOP03-DEL04-SC,-27.00218,-48.636433
1,MT,163,559.0,DIAMANTINO,SPRF-MT,DEL01-MT,UOP01-DEL01-MT,-14.24293501,-56.12516399
2,PB,101,60.2,SANTA RITA,SPRF-PB,DEL01-PB,UOP04-DEL01-PB,-7.00141407,-35.06459811
3,ES,101,270.3,SERRA,SPRF-ES,DEL02-ES,UOP01-DEL02-ES,-20.22662056,-40.27075088
4,MS,158,79.2,PARANAIBA,SPRF-MS,DEL08-MS,UOP01-DEL08-MS,-19.55764424,-51.24892432
5,GO,20,181.0,SIMOLANDIA,SPRF-DF,DEL02-DF,UOP03-DEL02-DF,-14.470801,-46.486183
6,PR,376,394.0,IMBAU,SPRF-PR,DEL03-PR,UOP02-DEL03-PR,-24.55402879,-50.68691653
7,RJ,101,519.0,ANGRA DOS REIS,SPRF-RJ,DEL03-RJ,UOP03-DEL03-RJ,-23.00804213,-44.44745093
8,BA,101,208.0,GOVERNADOR MANGABEIRA,SPRF-BA,DEL01-BA,UOP02-DEL01-BA,-12.60137779,-39.05031323
9,SC,101,158.0,PORTO BELO,SPRF-SC,DEL04-SC,UOP03-DEL04-SC,-27.19096301,-48.61259017
